# Phase 2 — ArcFace 1:1 verification
## Goal
Compare a fresh probe with only the explicitly claimed pseudonym. No matching before adequate real development calibration; no identity search. Scores are unit-normalized cosine similarities in [-1,1]. A match does not establish liveness or authorize attendance.

## Setup

In [ ]:
from pathlib import Path
import sys, json, os, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/phase2_verification.yaml').exists())
sys.path.insert(0,str(ROOT/'src'))
os.environ['MPLCONFIGDIR'] = str(ROOT/'.cache/matplotlib')
from face_verification import *
config = load_config(root=ROOT)
REPORTS = ROOT/'reports/phase2'
# Real faces are opt-in. Set these manually only after confirming each participant's consent.
RUN_REAL_FACES = False
ADULT_CONSENT = False
IDENTITY_CONSENT = False
PRIVATE_MANIFEST = ROOT/'data/private/consent_manifest.json'
print('Interpreter:',sys.executable)
print('Real-face workflow enabled:',RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT)


PRIVATE_INPUTS = Path(os.environ.get('FACEATTEND_PRIVATE_INPUTS', str(Path.home()/'AppData/Local/FaceAttend/phase2/private')))
# Explicit executor opt-in; ordinary Run All keeps all defaults false.
if os.environ.get('FACEATTEND_CONSENTED_SMOKE') == '1':
    supplied = json.loads((PRIVATE_INPUTS/'input_manifest.json').read_text())
    assert supplied['adult_identity_verification_consent_confirmed_by_user'] is True
    assert supplied['research_terms_confirmed_by_user'] is True
    RUN_REAL_FACES = ADULT_CONSENT = IDENTITY_CONSENT = True
print('Consented three-image smoke enabled:',RUN_REAL_FACES)


## Synthetic similarity checks

In [ ]:
from face_verification.quality import cosine_similarity
print({'identical':cosine_similarity([1,0],[2,0]),'orthogonal':cosine_similarity([1,0],[0,1]),'opposite':cosine_similarity([1,0],[-1,0])})

## Uncalibrated refusal with injected synthetic vectors
The isolated configuration has no threshold artifact. It must refuse before calling the detector; a mocked perfect embedding cannot bypass calibration.

In [ ]:
from tempfile import TemporaryDirectory
from cryptography.fernet import Fernet
from types import SimpleNamespace
from dataclasses import replace
from face_verification.schemas import FaceResult
import numpy as np
with TemporaryDirectory(dir=ROOT/'.cache/tmp') as temporary:
    isolated = replace(config,root=Path(temporary))
    store = TemplateStore(isolated.path(isolated.store_dir),key=Fernet.generate_key())
    fake_models = SimpleNamespace(config=isolated,model_id='synthetic_test',embedding_dim=3)
    samples = [np.full((200,200,3),100+i,np.uint8) for i in range(3)]
    enroll_user('volunteer_001',samples,store,fake_models,adult_consent=True,identity_consent=True,
        detector=lambda image,models:FaceResult(Status.OK,aligned=image),embedder=lambda *a:np.array([1,0,0]))
    def must_not_infer(*args):
        raise AssertionError('Uncalibrated inference was attempted.')
    result = verify_claimed_identity('volunteer_001',None,store,fake_models,adult_consent=True,identity_consent=True,detector=must_not_infer)
    print(json.dumps(result.to_dict(),indent=2))
    assert result.status == Status.THRESHOLD_NOT_CALIBRATED and not result.verified

## Real claimed-ID verification — manual opt-in

In [ ]:
if RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT:
    import cv2
    models = initialize_models(config)
    with TemporaryDirectory(dir=ROOT/'.cache/tmp') as temporary:
        empty_store = TemplateStore(Path(temporary),key=Fernet.generate_key())
        image = cv2.imread(str(PRIVATE_INPUTS/'captures/volunteer_003_input.jpeg'))
        result = verify_claimed_identity('volunteer_003',image,empty_store,models,adult_consent=True,identity_consent=True)
        print({'status':result.status.value,'verified':result.verified,'threshold':result.threshold})
        assert result.status == Status.UNKNOWN_ID and not result.verified
else:
    print('NOT_RUN: real verification requires consented fresh probes and enrollment.')

## Checks and next steps
The real input receives UNKNOWN_ID because one photo cannot establish an enrollment. The synthetic enrolled-ID example separately demonstrates THRESHOLD_NOT_CALIBRATED. Actual images are not compared with invented references or synthetic thresholds. No participant is marked verified.

Collect proper enrollment and independent genuine probes first. A match would still not prove liveness.

## Latest same-session probe scores and uncalibrated refusals

This section reads the executed ten-image retest artifact. The initial three-image experiment above is retained as history. User-supplied labels are represented by pseudonyms; no portraits or vectors are displayed. The user confirmed all ten captures are from one session. Enrollment uses photos 1/2/4 and 7/8/9; probes use 3/5 and 6/10, selected before score inspection. The encrypted enrollment smoke used a process-only key and removed all templates afterward.

These results establish implementation behavior, not a calibrated matching decision, independent-session accuracy, liveness, FMR/FNMR reliability or production readiness.

In [ ]:
batch = json.loads((REPORTS/'batch02_results.json').read_text())
print(json.dumps({'diagnostic_comparisons':batch['same_session_probe_comparisons'],
    'actual_verification_api':batch['verification_attempts']},indent=2))
assert all(a['status']=='THRESHOLD_NOT_CALIBRATED' and not a['verified'] for a in batch['verification_attempts'])
assert all(p['decision'] is None and p['used_for_calibration'] is False for p in batch['same_session_probe_comparisons'])

## Latest single-subject cross-session diagnostic

This section reads `batch03_results.json`, produced by an actual fresh-process GPU run. The user supplied one subject's ten photos and confirmed every photo came from a separate session; exact dates were not supplied. Enrollment photos 1/2/7 and probe photos 3/4/5/6/8/9/10 were fixed before inspecting scores. Photos 6/10 fail the unchanged size/clipping gate. No rejected image is embedded.

The three-image encrypted enrollment, duplicate-ID rejection and deletion passed. Five valid probes have diagnostic genuine scores 0.529–0.835; ten comparisons with explicitly labeled prior controls have scores 0.022–0.268. Session independence is user-confirmed for the new subject; cross-subject chronology is unconfirmed. Comparisons share subjects/templates, so the counts are dependent evidence, not independent population trials. All operational verification attempts refuse with THRESHOLD_NOT_CALIBRATED. No production threshold, FMR/FNMR, liveness or attendance claim is issued.

Names, original images and vectors remain out of public notebook outputs. Test templates and the process-only key were removed. Earlier cohorts above are preserved as history.

In [ ]:
latest = json.loads((REPORTS/'batch03_results.json').read_text())
print(json.dumps({'probe_diagnostics':latest['descriptive_probe_comparisons'],'verification_api_attempts':latest['verification_attempts']},indent=2))
assert all(a['status']=='THRESHOLD_NOT_CALIBRATED' and not a['verified'] for a in latest['verification_attempts'])